In [ ]:
import pandas as pd
import os
import numpy as np
import json

In [ ]:
tcga_path = "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/TCGA/"

In [ ]:
# Define file paths
clinical_file = tcga_path + "clinical.tsv"
followup_file = tcga_path + "follow_up.tsv"
sample_sheet_file = tcga_path + "gdc_sample_sheet.2025-09-22.tsv"

# Process clinical.tsv for OS data 
print(f"Processing {clinical_file} for OS data...")
# must use 'cases.submitter_id' as the linking key
clinical_df = pd.read_csv(clinical_file, sep='\t', usecols=[
    'cases.submitter_id',
    'demographic.vital_status',
    'demographic.days_to_death'
], dtype=str)

# Replace '--' with NaN
clinical_df.replace("'--", np.nan, inplace=True)

# Keep only one record per case, using submitter_id
clinical_df.drop_duplicates(subset=['cases.submitter_id'], keep='first', inplace=True)
clinical_df.set_index('cases.submitter_id', inplace=True) # Set index for joining
print(f"Loaded {clinical_df.shape[0]} unique case records from clinical.tsv.")

# Process follow_up.tsv for PFS and Last Follow-up
print(f"Processing {followup_file} for PFS and last follow-up data...")
# must use 'cases.submitter_id' as the linking key
followup_df = pd.read_csv(followup_file, sep='\t', usecols=[
    'cases.submitter_id',
    'follow_ups.days_to_progression',
    'follow_ups.days_to_follow_up'
], dtype=str)

# Replace '--' with NaN
followup_df.replace("'--", np.nan, inplace=True)

# Convert day columns to numeric
followup_df['follow_ups.days_to_progression'] = pd.to_numeric(followup_df['follow_ups.days_to_progression'])
followup_df['follow_ups.days_to_follow_up'] = pd.to_numeric(followup_df['follow_ups.days_to_follow_up'])

# Sort by days_to_follow_up to get the latest record
followup_df.sort_values(by='follow_ups.days_to_follow_up', ascending=False, inplace=True)

# Group by 'cases.submitter_id' and take the first (latest) record
latest_followup_df = followup_df.groupby('cases.submitter_id').first()
print(f"Loaded {latest_followup_df.shape[0]} unique latest follow-up records.")

# Merge Clinical and Follow-up Data 
print("Merging OS and PFS/follow-up data on 'cases.submitter_id'...")
merged_clinical_df = clinical_df.join(latest_followup_df, how='outer')

# Calculate Final OS and PFS Variables
print("Calculating OS and PFS status and time variables...")

# Convert days_to_death to numeric
merged_clinical_df['demographic.days_to_death'] = pd.to_numeric(merged_clinical_df['demographic.days_to_death'])

# Fill NaN in follow-up time with 0 
merged_clinical_df['follow_ups.days_to_follow_up'].fillna(0, inplace=True)

# OS Calculation
merged_clinical_df['OS_STATUS'] = np.where(
    merged_clinical_df['demographic.vital_status'] == 'Dead', 1, 0
)
merged_clinical_df.loc[merged_clinical_df['demographic.vital_status'].isna(), 'OS_STATUS'] = 0 

# OS_TIME: Use days_to_death if Dead. If Alive, use days_to_last_follow_up.
# Use combine_first to take 'days_to_death' if it exists, otherwise use 'days_to_follow_up'
merged_clinical_df['OS_TIME'] = merged_clinical_df['demographic.days_to_death'].combine_first(merged_clinical_df['follow_ups.days_to_follow_up'])
# Correct for 'Alive' patients: their time is only the follow-up time.
merged_clinical_df.loc[merged_clinical_df['demographic.vital_status'] == 'Alive', 'OS_TIME'] = merged_clinical_df['follow_ups.days_to_follow_up']


# PFS Calculation 
merged_clinical_df['PFS_STATUS'] = np.where(
    merged_clinical_df['follow_ups.days_to_progression'].notna(), 1, 0
)

# PFS_TIME: Use days_to_progression if it exists. If not, use days_to_last_follow_up.
merged_clinical_df['PFS_TIME'] = merged_clinical_df['follow_ups.days_to_progression'].combine_first(merged_clinical_df['follow_ups.days_to_follow_up'])

# Select final clinical columns
final_clinical_data = merged_clinical_df[[
    'OS_STATUS', 'OS_TIME', 'PFS_STATUS', 'PFS_TIME'
]]

# Drop any cases with no valid time data
final_clinical_data = final_clinical_data[
    (final_clinical_data['OS_TIME'] > 0) | (final_clinical_data['PFS_TIME'] > 0)
]
print(f"Created final clinical matrix for {final_clinical_data.shape[0]} cases with valid time data.")

# Process Sample Sheet for Primary Tumors 
print(f"Processing {sample_sheet_file} to find primary tumor samples...")
sample_sheet_df = pd.read_csv(sample_sheet_file, sep='\t')

# Filter for "Primary" in "Tumor Descriptor"
primary_tumor_samples = sample_sheet_df[
    sample_sheet_df['Tumor Descriptor'] == 'Primary'
].copy()

# Select only the file name and case ID
# 'Case ID' in this file is the submitter_id
primary_tumor_map = primary_tumor_samples[['File Name', 'Case ID']]
primary_tumor_map.rename(columns={
    'File Name': 'file_name',
    'Case ID': 'cases.submitter_id' # Rename to match the clinical data key
}, inplace=True)
print(f"Found {primary_tumor_map.shape[0]} primary tumor samples.")

# Create the Final File-to-Clinical Map 
print("Creating final map from file_name to clinical data...")

final_map_df = primary_tumor_map.merge(
    final_clinical_data,
    on='cases.submitter_id', 
    how='left' 
)

# Set 'file_name' as the index for the final merge
final_map_df.set_index('file_name', inplace=True)
final_map_df.drop(columns=['cases.submitter_id'], inplace=True)

# Drop samples that didn't have any matching clinical data
final_map_df.dropna(subset=['OS_STATUS', 'OS_TIME', 'PFS_STATUS', 'PFS_TIME'], how='all', inplace=True)

print(f"Final map created with {final_map_df.shape[0]} samples with clinical data.")

# Save this map to a CSV for inspection and future use
final_map_df.to_csv("file_to_clinical_survival_map.csv")
print("\nSaved file-to-clinical map to 'file_to_clinical_survival_map.csv'")

# Load Expression Data

all_counts = []  # list to hold per-sample Series

for file in os.listdir(tcga_path):
    if os.path.isdir(os.path.join(tcga_path, file)):
        # find the tsv file
        try:
            tsv_path_obj = [tsv_path for tsv_path in os.listdir(os.path.join(tcga_path, file)) if tsv_path.endswith(".tsv")][0]
            counts = pd.read_csv(os.path.join(tcga_path, file, tsv_path_obj), sep="\t", header=1)[["gene_name", "unstranded"]]      
            # turn into Series with gene_name as index, sample name as column
            sample_series = counts.set_index("gene_name")["unstranded"]

            sample_series.name = tsv_path_obj 

            all_counts.append(sample_series)

        except:
            print(f"No tsv file found in {file}, but found:")
            print([tsv_path for tsv_path in os.listdir(os.path.join(tcga_path, file))])


# concatenate all samples side-by-side
final_df = pd.concat(all_counts, axis=1)
final_df_transposed = final_df.T


# Final Merge: Expression + Clinical Data 

try:
    print(f"Merging expression data ({final_df_transposed.shape}) with clinical map ({final_map_df.shape})...")

    final_merged_data = final_df_transposed.join(
        final_map_df,
        how='inner'
    )
    
    print(f"Final merged DataFrame shape: {final_merged_data.shape}")

    # Clean the index
    print("\nCleaning index ...")
    if not final_merged_data.empty:
        final_merged_data.index = final_merged_data.index.str.split('.').str[0]
        print("Head after index cleaning:")
        print(final_merged_data.head())
    

    else:
        print("Final merged data is empty. No matches found between expression files and clinical map.")

except NameError:
    print("\n'final_df_transposed' not found. Skipping final merge.")
    print("Please run your expression loading code before this section.")
except Exception as e:
    print(f"An error occurred during final merge: {e}")

In [ ]:
final_merged_data = final_merged_data.iloc[:, 4:]

In [ ]:
final_merged_data.to_csv(tcga_path + "final_expression_with_survival.csv")

In [ ]:
final_merged_data.to_csv("final_expression_with_survival.csv")

In [ ]:
final_map_df.to_csv(tcga_path + "file_to_clinical_survival_map.csv")